In [10]:
secfiles = [
    "sections-data-tool2-shumei.json",
    "sections-shumei-seqingyouxi.json",
    "sections-data-tool-shumei.json",
    "sections-nsfw.json",
    "sections-nsfw-shumei.json",
    "sections-safety240722.json",
]

import json

secdata = {}
seccounts = {}

for sf in secfiles:
    data = json.load(open(sf))
    secdata[sf] = data

    counts = seccounts[sf] = {}

    for k, v in data["count"].items():
        counts[k] = v

from pprint import pprint

pprint(seccounts)

{'sections-data-tool-shumei.json': {'ban': 4886,
                                    'blackandwhitelist': 21,
                                    'minor': 195,
                                    'normal': 118249,
                                    'politics': 24303,
                                    'porn': 4702,
                                    'sexy': 0,
                                    'star': 347,
                                    'violence': 923},
 'sections-data-tool2-shumei.json': {'normal': 181292,
                                     'politics': 82035,
                                     'porn': 903},
 'sections-nsfw-shumei.json': {'normal': 3111, 'porn': 4408},
 'sections-nsfw.json': {'drawings': 9646,
                        'hentai': 2597,
                        'neutral': 23629,
                        'porn': 355,
                        'sexy': 4176},
 'sections-safety240722.json': {'20240322_8k_nsfw_dup_1507': 1507,
                                '2024032

In [15]:
import random
from copy import deepcopy

collects = {
    # 0930
    "sections-data-tool2-shumei.json": {
        "normal": {"train": 0.7, "val": 400, "label": "normal"},
        "politics": {"train": 2.0, "val": 500, "label": "politics"},
        # "porn": {"train": 1.0, "val": 500, "label": "porn"},
    },
    "sections-shumei-seqingyouxi.json": {
        "normal": {"train": 0.7, "val": 00, "label": "normal"},
        "porn": {"train": 10.0, "val": 00, "label": "porn"},
    },
    # 以前
    "sections-data-tool-shumei.json": {
        "normal": {"train": 0.7, "val": 400, "label": "normal"},
        "politics": {"train": 2.0, "val": 500, "label": "politics"},
        "porn": {"train": 10.0, "val": 500, "label": "porn"},
    },
    "sections-nsfw-shumei.json": {
        "normal": {"train": 0.7, "val": 100, "label": "normal"},
        "porn": {"train": 10.0, "val": 500, "label": "porn"},
    },
    "sections-nsfw.json": {
        "drawings": {"train": 0.7, "val": 50, "label": "normal"},
        "neutral": {"train": 0.7, "val": 50, "label": "normal"},
    },
    "sections-safety240722.json": {
        "20240322_8k_nsfw_dup_1507": {"train": 1.0, "val": 0, "label": "porn"},
        "20240329_163k_nsfw_dup_110k": {"train": 1.0, "val": 0, "label": "porn"},
        "nsfw_23k": {"train": 1.0, "val": 0, "label": "porn"},
        "sq_8k": {"train": 1.0, "val": 0, "label": "porn"},
    },
}

train_counts = {}
val_counts = {}
train_anns = []
val_anns = []
labels = []
label_map = {}

random.seed(0)

for sf, col in collects.items():
    # for each file, collect multiple sections
    counts = seccounts[sf]
    data = secdata[sf]

    for k, v in col.items():
        # collect sections

        label = v["label"]
        val_count = int(v.get("val",0))
        train_count = int((counts[k] - val_count) * v["train"])

        # stat
        if label in train_counts:
            train_counts[label] += train_count
        else:
            train_counts[label] = train_count
        if label in val_counts:
            val_counts[label] += val_count
        else:
            val_counts[label] = val_count
        if label not in labels:
            labels.append(label)
            label_map[label] = len(labels) - 1
        label_id = label_map[label]

        # collect data
        files = deepcopy(data["files"][k])
        random.shuffle(files)

        # get val first to avoid overlap
        for _ in range(val_count):
            val_anns.append((files.pop(), label_id))

        # sample train
        files = files * int(v["train"] + 1)
        files = files[: int(train_count)]

        train_ann_ = [(f, label_id) for f in files]
        train_anns.extend(train_ann_)

pprint(train_counts)
pprint(val_counts)
pprint(train_anns[::len(train_anns)//20])
pprint(val_anns[::len(val_anns)//20])

assert set(train_anns) & set(val_anns) == set()

{'normal': 234921, 'politics': 210676, 'porn': 214697}
{'normal': 1000, 'politics': 1000, 'porn': 1000}
[('data_tool2/res2/陈光诚/google_陈光诚/00348.jpeg', 0),
 ('data_tool2/res2/新公民运动/google_新公民运动/00120.jpeg', 0),
 ('data_tool2/res2/多名女性/google_多名女性/00453.jpeg', 0),
 ('data_tool2/res2/梦鸽/google_梦鸽/00512.jpeg', 0),
 ('data_tool2/res2/蔡英文/google_蔡英文/00020.jpeg', 1),
 ('data_tool2/res2/亲自指挥/google_亲自指挥/00262.jpeg', 1),
 ('data_tool2/res3/中共/google_中共/00092.jpeg', 1),
 ('data_tool2/res2/人民/google_人民/00647.jpeg', 1),
 ('data_tool2/res2/文工团/google_文工团/00411.jpeg', 1),
 ('pachong2/huang/seqingyouxiguanggao/1300037569.png', 2),
 ('data_tool/res/反动分裂/google_反动媒体标志/00134.jpeg', 0),
 ('data_tool/res/负面事件/google_吸毒人物/00806.jpeg', 0),
 ('data_tool/res/领导人/google_国外领导/00109.jpeg', 1),
 ('data_tool/res/男裸露/google_男下露点/00139.png', 2),
 ('data_tool/res/男裸露/google_男内裤特写/00389.jpeg', 2),
 ('raw_data/hentai/IMAGES/am1rr7f6t0z21.jpg', 2),
 ('raw_data/neutral/IMAGES/pgxl4sn1om531.jpg', 0),
 ('safety240722/seqin

In [12]:
with open('huangfan-shumei-1009-train.txt', 'w') as f:
    for ann in train_anns:
        f.write(f"{ann[0]} {ann[1]}\n")

with open('huangfan-shumei-1009-val.txt', 'w') as f:
    for ann in val_anns:
        f.write(f"{ann[0]} {ann[1]}\n")

with open('huangfan-shumei-1009-labels.txt', 'w') as f:
    for label in labels:
        f.write(f"{label}\n")